# Context Managers

Chapter 10 showed three ways to close a file:

```python
f = open("data.txt")      # 1. remember to call f.close()
...

try:                      # 2. try/finally
    f = open("data.txt")
finally:
    f.close()

with open("data.txt") as f:   # 3. the professional way
    ...
```

It called the third one "the professional way to handle files", and we have used it ever since — in chapter 10 for text, CSV and JSON, and in chapter 12 for the 200,000-row lazy pipeline.

But we never said **what `with` actually does**. It looks like special file syntax. It is not. `with` is a general protocol that any object can implement, and files are just the first thing you met that implements it.

This chapter is where chapters 12 and 13 pay off together: the shortest way to write a context manager is a **generator** wrapped in a **decorator**.

**What we will learn:**

1. Why cleanup is harder than it looks
2. The protocol behind `with`: `__enter__` and `__exit__`
3. Writing your own context manager as a class
4. What `as` really binds, and what `__exit__` sees when the block crashes
5. `@contextlib.contextmanager` — the six-line version
6. The `contextlib` tools worth knowing: `suppress`, `closing`, `redirect_stdout`
7. Real patterns: block timers, rollbacks, atomic file writes

### Setting Up

Like chapter 10, we will keep our practice files in their own folder so nothing clutters your working directory.

In [1]:
import os
from pathlib import Path

os.makedirs("context_demo", exist_ok=True)

with open("context_demo/sales.csv", "w", encoding="utf-8") as f:
    f.write("region,units\n")
    f.write("north,120\n")
    f.write("south,95\n")
    f.write("east,140\n")
    f.write("west,88\n")

print("Sample files ready.")

Sample files ready.


---
# 1. The Problem: Cleanup That Must Happen

When you open a file, the operating system hands your program a **file descriptor** — a limited resource. Chapter 10 said you must give it back with `close()`. Here is what happens when something goes wrong before you get the chance.

In [2]:
f = open("context_demo/sales.csv", encoding="utf-8")

try:
    header = f.readline()
    units = int("not a number")     # a bad row in the data -- this raises
    f.close()                       # never reached
except ValueError as e:
    print("Error:", e)

print("Is the file still open?", not f.closed)
f.close()                           # clean up so the rest of the notebook is tidy

Error: invalid literal for int() with base 10: 'not a number'
Is the file still open? True


The `close()` call was written. It simply never ran, because the exception jumped straight past it to the `except` block.

One leaked file on your laptop is harmless. A web server that leaks one file per request runs out of descriptors and stops accepting connections — this is a real and common outage cause.

Chapter 9 gave us the fix: `finally` always runs.

In [3]:
f = open("context_demo/sales.csv", encoding="utf-8")

try:
    header = f.readline()
    units = int("not a number")
except ValueError as e:
    print("Error:", e)
finally:
    f.close()                       # runs no matter what

print("Is the file still open?", not f.closed)

Error: invalid literal for int() with base 10: 'not a number'
Is the file still open? False


That is correct — and it is four lines of bookkeeping around one line of work. Worse, you have to remember to write it **every single time**, and the thing you must remember (`f.close()`) is written far away from the thing that created the obligation (`open()`).

`with` moves the obligation into the object itself:

In [4]:
with open("context_demo/sales.csv", encoding="utf-8") as f:
    header = f.readline()
    # imagine the same crash happening here

print("Is the file still open?", not f.closed)

Is the file still open? False


**The pattern is not about files.** It shows up everywhere a resource has to be given back:

| You acquire | You must release | If you forget |
|---|---|---|
| An open file | `close()` | Leaked descriptors; unflushed writes |
| A database connection | `close()` | The connection pool drains and the app stalls |
| A lock | `release()` | Every other thread waits forever |
| A changed setting | Restore the old value | The rest of the program silently misbehaves |
| A started transaction | `commit()` or `rollback()` | Half-written data |
| A temporary directory | Delete it | Disk fills up |

Every one of these is a **setup / teardown pair where the teardown must happen even if the middle part crashes**. That is exactly what a context manager is for.

---
# 2. What `with` Actually Does

`with` is not file syntax. It is a **protocol**: any object that defines two methods, `__enter__` and `__exit__`, can be used in a `with` statement.

Like `__init__` from chapter 8 and `__iter__`/`__next__` from chapter 12, these are dunder methods Python calls for you.

This:

```python
with open("data.txt") as f:
    body
```

is roughly shorthand for this:

```python
manager = open("data.txt")
f = manager.__enter__()
try:
    body
finally:
    manager.__exit__(exception_type, exception_value, traceback)
```

So the `try/finally` we wrote by hand did not disappear — Python writes it for you, and the object supplies the cleanup.

First, proof that a file object really has those methods:

In [5]:
f = open("context_demo/sales.csv", encoding="utf-8")

print("has __enter__:", hasattr(f, "__enter__"))
print("has __exit__ :", hasattr(f, "__exit__"))

f.close()

has __enter__: True
has __exit__ : True


And we can drive the protocol by hand, with no `with` statement anywhere:

In [6]:
f = open("context_demo/sales.csv", encoding="utf-8")

same_file = f.__enter__()                       # what 'as' would have bound
print("__enter__ returned the file itself:", same_file is f)
print("first line:", same_file.readline().strip())

f.__exit__(None, None, None)                    # the three arguments come later
print("closed:", f.closed)

__enter__ returned the file itself: True
first line: region,units
closed: True


**Never write that in real code** — it is here only to show that `with` is ordinary method calls, not magic.

And if an object does *not* implement the protocol, `with` says so plainly:

In [7]:
try:
    with 42 as number:                  # type: ignore -- deliberately wrong
        print(number)
except TypeError as e:
    print("TypeError:", e)

TypeError: 'int' object does not support the context manager protocol


That error message is worth memorising. It means "this object has no `__enter__`/`__exit__`", which usually means you passed the wrong thing — a very common mistake is `with open` versus `with my_path` when `my_path` is just a string.

---
# 3. Writing Your Own Context Manager

Two methods, and you are done. The simplest useful one just announces when a block starts and ends:

In [8]:
class Section:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"[start] {self.name}")
        return self                     # this is what 'as' binds

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"[end]   {self.name}")


with Section("loading data"):
    print("   ...reading rows...")
    print("   ...cleaning rows...")

print("back outside the block")

[start] loading data
   ...reading rows...
   ...cleaning rows...
[end]   loading data
back outside the block


Read the output top to bottom and you can see the order: `__enter__`, then the body, then `__exit__`, then the code after the block.

Now something closer to real work. Chapter 9 built a `DatabaseConnection` class and wrapped every use of it in `try/finally` to guarantee the connection closed. Here is that same class with the guarantee moved inside it:

In [9]:
class Database:
    # A stand-in for a real driver such as sqlite3 or psycopg2.

    def __init__(self, name):
        self.name = name
        self.queries = []

    def __enter__(self):
        print(f"[DB] connected to {self.name!r}")
        return self

    def run(self, query):
        self.queries.append(query)
        print(f"[DB] running: {query}")

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"[DB] closed after {len(self.queries)} queries")


with Database("analytics") as db:
    db.run("SELECT count(*) FROM users")
    db.run("SELECT * FROM orders LIMIT 10")

[DB] connected to 'analytics'
[DB] running: SELECT count(*) FROM users
[DB] running: SELECT * FROM orders LIMIT 10
[DB] closed after 2 queries


Compare that with chapter 9's version, where every caller had to remember the `try/finally`. Here the caller writes one line and cannot get it wrong.

The real test is what happens when the block crashes:

In [10]:
try:
    with Database("analytics") as db:
        db.run("SELECT * FROM users")
        db.run("UPDATE users SET last_seen = now()")
        raise RuntimeError("network dropped mid-query")
except RuntimeError as e:
    print("caught outside the block:", e)

[DB] connected to 'analytics'
[DB] running: SELECT * FROM users
[DB] running: UPDATE users SET last_seen = now()
[DB] closed after 2 queries
caught outside the block: network dropped mid-query


Look at the order: **`[DB] closed` printed first**, then our `except` ran. The connection was closed on the way out, and the exception still reached the caller — nothing was hidden.

That is the default and it is the right default. A context manager's job is to clean up, not to decide that your error did not matter.

---
# 4. What `as` Really Binds

`as name` binds **whatever `__enter__` returns** — not the manager, not the object you passed in. Most managers `return self`, which is why it usually looks like the same object.

Forget that `return` and you get one of the most confusing beginner bugs in Python:

In [11]:
class Broken:
    def __enter__(self):
        print("opening")
        # oops -- no return statement

    def __exit__(self, exc_type, exc_value, traceback):
        print("closing")


with Broken() as thing:
    print("thing is:", thing)

opening
thing is: None
closing


`__enter__` returned `None` (every function without a `return` does), so `thing` is `None`. Nothing has gone wrong *yet* — the failure arrives at the next line that tries to use it:

In [12]:
try:
    with Broken() as thing:
        thing.run("SELECT 1")           # type: ignore -- thing is None
except AttributeError as e:
    print("AttributeError:", e)

opening
closing
AttributeError: 'NoneType' object has no attribute 'run'


`'NoneType' object has no attribute ...` inside a `with` block is almost always a missing `return self`.

But returning `self` is a convention, not a rule. `__enter__` can hand the block anything that is useful — here it hands over a list to collect into, and reports on it afterwards:

In [13]:
class Collected:
    def __enter__(self):
        self.items = []
        return self.items               # the block gets the list, not the manager

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"collected {len(self.items)} items: {self.items}")


with Collected() as bucket:
    bucket.append("north")
    bucket.append("south")
    bucket.append("east")

collected 3 items: ['north', 'south', 'east']


---
# 5. `__exit__` and Exceptions

We have been writing `__exit__(self, exc_type, exc_value, traceback)` and ignoring the three arguments. Here is what they are for.

When the block finishes **cleanly**, all three are `None`. When the block **raises**, they are the exception's class, the exception object itself, and its traceback — the same three things `sys.exc_info()` returns, mentioned in chapter 9.

In [14]:
class Watcher:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("__exit__: block finished cleanly")
        else:
            print(f"__exit__: block raised {exc_type.__name__} -> {exc_value}")


with Watcher():
    print("doing fine")

print()

try:
    with Watcher():
        total = 100 / 0
except ZeroDivisionError:
    print("...and it still reached us out here")

doing fine
__exit__: block finished cleanly

__exit__: block raised ZeroDivisionError -> division by zero
...and it still reached us out here


So `__exit__` gets to *see* the exception. That is what makes rollbacks possible: the manager can tell the difference between "the block succeeded, commit" and "the block failed, undo everything".

### Returning `True` swallows the exception

`__exit__` has one more power. If it returns a **truthy** value, Python treats the exception as handled and it never leaves the `with` block.

In [15]:
class Swallow:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("__exit__ saw:", exc_type.__name__)
        return True                     # "I handled it"


with Swallow():
    print("about to fail")
    result = 100 / 0
    print("this line never runs")

print("we get here -- the exception never escaped the block")

about to fail
__exit__ saw: ZeroDivisionError
we get here -- the exception never escaped the block


Use this carefully. `Swallow` does not swallow *the error you were thinking of* — it swallows **every** error, including the ones you never anticipated:

In [16]:
with Swallow():
    total = int("120") + int("nrth")    # a typo in a region name
    print("total:", total)

print("no crash, no total, no clue anything went wrong")

__exit__ saw: ValueError
no crash, no total, no clue anything went wrong


A typo silently produced nothing at all. If that block had been computing a monthly revenue figure, the report would simply be missing a number and no one would know why.

The honest version swallows exactly one kind of error and re-raises everything else. Note the shape: **`__exit__` returns a boolean answer to "should this exception stop here?"**

In [17]:
class IgnoreMissingFile:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            return False
        if issubclass(exc_type, FileNotFoundError):
            print(f"ignoring: {exc_value.filename} was already gone")
            return True                 # handled
        return False                    # anything else -- let it through


with IgnoreMissingFile():
    os.remove("context_demo/never_existed.txt")

print("cleanup finished\n")

try:
    with IgnoreMissingFile():
        os.remove(12345)                # type: ignore -- a bug: not a filename
except TypeError as e:
    print("TypeError got through, as it should:", e)

ignoring: context_demo/never_existed.txt was already gone
cleanup finished

TypeError got through, as it should: remove: path should be string, bytes or os.PathLike, not int


### The accidental swallow

Because *any* truthy value counts, this bug is easy to write without noticing. `__exit__` ends with a `return` that was meant to be informational:

In [18]:
class Cleanup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        removed = ["batch_1.tmp", "batch_2.tmp"]
        print(f"cleaned up {len(removed)} temp files")
        return len(removed)             # meant as a count -- but 2 is truthy!


with Cleanup():
    1 / 0

print("the ZeroDivisionError vanished, and nothing said so")

cleaned up 2 temp files
the ZeroDivisionError vanished, and nothing said so


**Rule:** an `__exit__` should end in `return True`, `return False`, or no `return` at all. Anything else is asking for a swallowed bug. If you have nothing to suppress, do not return a value — `None` is falsy, which is exactly what you want.

---
# 6. A Timer for Blocks of Code

Chapter 13 built a `@execution_timer` decorator. It has one limitation: it times a **whole function**. If a function does three things and only one of them is slow, the decorator cannot tell you which.

A context manager times a **block**, which can be any size you like:

In [19]:
import time

class Timer:
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self.start
        print(f"{self.label}: {self.elapsed:.1f}s")


with Timer("loading sales data"):
    time.sleep(0.5)                     # stand-in for the real work

loading sales data: 0.5s


Now the missing half of the decorator — timing the *parts* of one function:

In [20]:
def build_report(rows):
    with Timer("  parse    "):
        time.sleep(0.2)
        parsed = [row.split(",") for row in rows]

    with Timer("  aggregate"):
        time.sleep(0.5)                 # the slow step
        total = sum(int(cells[1]) for cells in parsed)

    return total


with open("context_demo/sales.csv", encoding="utf-8") as f:
    next(f)                             # skip the header row
    rows = [line.strip() for line in f]

with Timer("build_report"):
    print("total units:", build_report(rows))

  parse    : 0.2s


  aggregate: 0.5s
total units: 443
build_report: 0.7s


The output reads like a small profiler, and it tells you immediately that `aggregate` is the expensive step.

| | `@execution_timer` (ch. 13) | `with Timer(...)` (this chapter) |
|---|---|---|
| Times | a whole function | any block of code |
| Applied | once, at the definition | anywhere, as often as you like |
| Needs the code to be | a function | anything |
| Best for | "which function is slow?" | "which part of this function is slow?" |

They are not rivals. The same object can be both — but a decorator and a context manager are the natural shapes for those two questions, and knowing both means you always have the right one.

Note also that a class-based manager is **reusable**: `Timer("x")` can be entered again and again, and the instance keeps `self.elapsed` afterwards for you to read.

In [21]:
stage = Timer("reusable")

with stage:
    time.sleep(0.2)

with stage:
    time.sleep(0.5)

print(f"the instance still remembers the last run: {stage.elapsed:.1f}s")

reusable: 0.2s


reusable: 0.5s
the instance still remembers the last run: 0.5s


---
# 7. `@contextlib.contextmanager` — Generators Plus Decorators

Writing a class for every setup/teardown pair is a lot of ceremony. The standard library has a shortcut, and it is built out of the last two chapters:

- Chapter 12: a **generator** runs up to `yield`, pauses, and resumes where it left off.
- Chapter 13: a **decorator** wraps a function to give it new behaviour.

`contextlib.contextmanager` is a decorator that turns a generator function into a context manager. Everything **before** the `yield` is `__enter__`; everything **after** it is `__exit__`.

In [22]:
from contextlib import contextmanager


@contextmanager
def section(name):
    print(f"[start] {name}")            # __enter__
    yield                               # the with-block runs right here
    print(f"[end]   {name}")            # __exit__


with section("loading data"):
    print("   ...reading rows...")

print("back outside the block")

[start] loading data
   ...reading rows...
[end]   loading data
back outside the block


Same output as the `Section` class in section 3, in a third of the code.

The value you `yield` is what `as` binds — the generator's equivalent of `return self`:

In [23]:
@contextmanager
def collected():
    items = []
    yield items                         # handed to 'as'
    print(f"collected {len(items)} items: {items}")


with collected() as bucket:
    bucket.append("north")
    bucket.append("south")

collected 2 items: ['north', 'south']


### The bug: teardown that never runs

There is a trap here, and it is the single most common mistake with `@contextmanager`. Look at this timer — no `try` anywhere:

In [24]:
@contextmanager
def timer(label):
    start = time.perf_counter()
    yield
    print(f"{label}: {time.perf_counter() - start:.1f}s")   # after the yield


with timer("happy path"):
    time.sleep(0.2)

happy path: 0.2s


Works. Now make the block fail:

In [25]:
try:
    with timer("crashing block"):
        time.sleep(0.2)
        raise RuntimeError("boom")
except RuntimeError as e:
    print("caught:", e)

print("...and notice there is no timing line above")

caught: boom
...and notice there is no timing line above


**The teardown never ran.** When the block raises, Python throws that exception *into* the generator at the `yield` — chapter 12 showed a generator paused at its `yield` and resumed by `next()`; throwing an exception in is the other way of resuming one. The exception propagates out of the generator immediately, so the lines after `yield` are skipped.

The class version never had this problem: Python guarantees `__exit__` is called. To get the same guarantee from a generator, wrap the `yield` in `try/finally` yourself:

In [26]:
@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:                            # runs on the way out, always
        print(f"{label}: {time.perf_counter() - start:.1f}s")


try:
    with timer("crashing block"):
        time.sleep(0.2)
        raise RuntimeError("boom")
except RuntimeError as e:
    print("caught:", e)

crashing block: 0.2s
caught: boom


**Rule:** if the teardown must always happen, the `yield` goes inside a `try/finally`. Write it that way by default.

### Handling the exception in a generator manager

`try/except` around the `yield` is the generator's version of "`__exit__` returned `True`" — if you catch the exception and do not re-raise it, it stops there.

In [27]:
@contextmanager
def ignore_missing_file():
    try:
        yield
    except FileNotFoundError as e:
        print(f"ignoring: {e.filename} was already gone")


with ignore_missing_file():
    os.remove("context_demo/never_existed.txt")

print("cleanup finished")

ignoring: context_demo/never_existed.txt was already gone
cleanup finished


One more thing to know: a `@contextmanager` object is **single-use**. Entering it runs the generator to the `yield`; there is nothing left to run a second time.

In [28]:
cm = section("used once")

with cm:
    print("   first use is fine")

try:
    with cm:
        print("   second use")
except AttributeError as e:
    print("second use failed:", e)

[start] used once
   first use is fine
[end]   used once
second use failed: '_GeneratorContextManager' object has no attribute 'args'


The message is cryptic, but it means "this manager already consumed its generator". The fix is to **call the function again** each time — `with section("..."):`, not `cm = section("..."); with cm:` — which is what everyone writes naturally anyway.

A class-based manager, as we saw with `Timer`, has no such limit.

---
# 8. Class or Generator?

Both produce a context manager. Pick by what the block needs:

| Use a **class** when | Use `@contextmanager` when |
|---|---|
| The block needs methods on the object (`db.run(...)`) | The block just needs setup and teardown |
| You want to reuse one instance repeatedly | A fresh one per `with` is fine |
| You want state readable afterwards (`stage.elapsed`) | Nothing survives the block |
| You want to subclass or extend it | It is a one-off helper |
| The teardown logic is long | It fits in a few lines |

In practice, most context managers you write day to day are short helpers, so `@contextmanager` wins on volume. Most context managers you *use* from libraries — files, connections, locks — are classes, because they have methods.

---
# 9. The `contextlib` Toolkit

`contextlib` is more than the decorator. Three of its tools are worth knowing now.

### `suppress` — the `try/except/pass` replacement

Chapter 9 listed `contextlib.suppress` under "advanced tools" and moved on. Here it is. This:

```python
try:
    os.remove(path)
except FileNotFoundError:
    pass
```

becomes one line:

In [29]:
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("context_demo/does_not_exist.txt")

print("no crash -- the file was already gone, which is fine")

no crash -- the file was already gone, which is fine


It reads better and, unlike a bare `except: pass`, it names exactly what you are willing to ignore. You can list several: `suppress(FileNotFoundError, PermissionError)`.

**But keep the block to one statement.** `suppress` does not skip the failing line and continue — it ends the whole block, exactly like any other exception would:

In [30]:
for name in ["temp_a.txt", "temp_b.txt"]:
    with open(f"context_demo/{name}", "w", encoding="utf-8") as f:
        f.write("scratch\n")

with suppress(FileNotFoundError):
    os.remove("context_demo/temp_a.txt")        # exists -- removed
    os.remove("context_demo/gone.txt")          # missing -- block ends HERE
    os.remove("context_demo/temp_b.txt")        # never runs

print("temp_a.txt still there?", os.path.exists("context_demo/temp_a.txt"))
print("temp_b.txt still there?", os.path.exists("context_demo/temp_b.txt"))

os.remove("context_demo/temp_b.txt")            # clean up properly

temp_a.txt still there? False
temp_b.txt still there? True


`temp_b.txt` survived. Everyone expects `suppress` to work line by line at some point; it does not. One statement per `with suppress(...)`, or a loop with the `with` inside it.

### `closing` — for objects with `close()` but no protocol

Plenty of older libraries give you an object with a `close()` method that predates the `with` statement. `closing` wraps one so it can be used with `with` anyway:

In [31]:
from contextlib import closing


class LegacyCursor:
    # An old-style object: it has close(), but no __enter__/__exit__.

    def fetch(self):
        return [("north", 120), ("south", 95)]

    def close(self):
        print("[cursor] closed")


with closing(LegacyCursor()) as cursor:
    print("rows:", cursor.fetch())

rows: [('north', 120), ('south', 95)]
[cursor] closed


`closing(obj)` is simply a context manager whose `__exit__` calls `obj.close()`. Nothing more.

### `redirect_stdout` — catching what a library prints

Sometimes a library prints things you would rather capture: put in a log, hide from the user, or check in a test. `redirect_stdout` sends `print()` somewhere else for the duration of a block.

In [32]:
import io
from contextlib import redirect_stdout


def noisy_model_training():
    print("epoch 1/3  loss=0.412")
    print("epoch 2/3  loss=0.233")
    print("epoch 3/3  loss=0.187")
    return 0.187


buffer = io.StringIO()                  # an in-memory text file

with redirect_stdout(buffer):
    final_loss = noisy_model_training()  # its prints go into the buffer

print("nothing was printed by the call above")
print("final loss:", final_loss)
print("captured lines:", buffer.getvalue().strip().splitlines())

nothing was printed by the call above
final loss: 0.187
captured lines: ['epoch 1/3  loss=0.412', 'epoch 2/3  loss=0.233', 'epoch 3/3  loss=0.187']


This is worth remembering for chapter 19 — capturing output is how you test a function that prints instead of returning.

---
# 10. Several Managers in One `with`

You often need two at once — read from one file, write to another. Nesting works:

```python
with open("in.csv") as fin:
    with open("out.csv", "w") as fout:
        ...
```

but a single `with` takes a comma-separated list, and closes them in reverse order:

In [33]:
with open("context_demo/sales.csv", encoding="utf-8") as fin, \
     open("context_demo/big_regions.csv", "w", encoding="utf-8") as fout:

    fout.write(next(fin))               # copy the header across
    for line in fin:
        region, units = line.strip().split(",")
        if int(units) >= 100:
            fout.write(line)

print(Path("context_demo/big_regions.csv").read_text(encoding="utf-8"))

region,units
north,120
east,140



From Python 3.10 you can also wrap the list in parentheses, which is easier to read once you have three or more:

```python
with (
    open("sales.csv") as fin,
    open("clean.csv", "w") as fout,
    Timer("copy"),
):
    ...
```

Note that the last one has no `as` — you only write `as` when you need the value.

---
# 11. Real-World Examples

### Netflix: a transaction that rolls back

This is the pattern `__exit__`'s three arguments exist for. The block either finishes and we commit, or it raises and we put everything back the way it was.

In [34]:
class InsufficientFunds(Exception):
    pass


class Transaction:
    # Snapshot on the way in; restore the snapshot if the block fails.

    def __init__(self, account):
        self.account = account

    def __enter__(self):
        self.snapshot = dict(self.account)
        return self.account

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print("  committed:", self.account)
        else:
            self.account.clear()
            self.account.update(self.snapshot)
            print(f"  rolled back after {exc_type.__name__}:", self.account)
        # no return -- the exception is not ours to swallow


def charge(account, amount):
    with Transaction(account) as acct:
        acct["balance"] -= amount
        acct["charges"] += 1
        if acct["balance"] < 0:
            raise InsufficientFunds(f"balance would be {acct['balance']}")


wallet = {"balance": 100, "charges": 0}

print("charging 30:")
charge(wallet, 30)

print("\ncharging 500:")
try:
    charge(wallet, 500)
except InsufficientFunds as e:
    print("  refused:", e)

print("\nfinal state:", wallet)

charging 30:
  committed: {'balance': 70, 'charges': 1}

charging 500:
  rolled back after InsufficientFunds: {'balance': 70, 'charges': 1}
  refused: balance would be -430

final state: {'balance': 70, 'charges': 1}


The failed charge left **no trace** — not the balance, not the charge counter. Without the context manager, the `charges` increment would have stuck around, because it happened before the check that raised.

### Any script: a temporary working directory

`os.chdir()` changes the working directory for the whole process. Forgetting to change back breaks every relative path the program uses afterwards, in ways that are painful to debug.

In [35]:
@contextmanager
def working_directory(path):
    previous = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)              # even if the block explodes


print("before:", os.path.basename(os.getcwd()))

with working_directory("context_demo"):
    print("inside:", os.path.basename(os.getcwd()))
    print("sales.csv visible without a path?", os.path.exists("sales.csv"))

print("after :", os.path.basename(os.getcwd()))

before: Python-Programming-Code
inside: context_demo
sales.csv visible without a path? True
after : Python-Programming-Code


And the point of the `finally` — the directory is restored even when the block fails:

In [36]:
try:
    with working_directory("context_demo"):
        raise RuntimeError("something went wrong in here")
except RuntimeError as e:
    print("caught:", e)

print("still back where we started:", os.path.basename(os.getcwd()))

caught: something went wrong in here
still back where we started: Python-Programming-Code


### Any data pipeline: writing a file atomically

Chapter 10 wrote files directly. That has a hidden risk: if the program crashes halfway through writing, the old file is already gone and the new one is half-written. You have destroyed good data and replaced it with garbage.

The fix is to write to a temporary file and only put it in place once the writing succeeded.

In [37]:
@contextmanager
def atomic_write(path, encoding="utf-8"):
    temp_path = path + ".tmp"
    f = open(temp_path, "w", encoding=encoding)
    try:
        yield f                         # the block writes into the temp file
    except Exception:
        f.close()
        os.remove(temp_path)            # failed -- throw the partial file away
        raise                           # and let the caller know
    else:
        f.close()
        os.replace(temp_path, path)     # succeeded -- swap it into place

In [38]:
report = "context_demo/report.txt"

with atomic_write(report) as f:
    f.write("Regional report\n")
    f.write("north: 120\n")

print("first write:")
print(Path(report).read_text(encoding="utf-8"))

first write:
Regional report
north: 120



In [39]:
try:
    with atomic_write(report) as f:
        f.write("Regional report\n")
        f.write("north: " + str(int("nrth")))   # crashes mid-write
except ValueError as e:
    print("write failed:", e)

print("\nthe good file is untouched:")
print(Path(report).read_text(encoding="utf-8"))
print("no leftover .tmp file:", not os.path.exists(report + ".tmp"))

write failed: invalid literal for int() with base 10: 'nrth'

the good file is untouched:
Regional report
north: 120

no leftover .tmp file: True


Two things to notice. First, the old report survived a crash that happened *after* writing had begun. Second, `atomic_write` used `try/except/else` rather than `try/finally`, because success and failure need **different** cleanup — this is the `else` clause from chapter 9 doing exactly the job it was designed for.

---
# 12. Going Further: `ExitStack`

*Skip this section on a first read — you will know when you need it.*

A comma-separated `with` handles a fixed number of managers. `ExitStack` handles a number you only learn at runtime, and guarantees every one of them is closed even if opening the fourth one fails.

In [40]:
from contextlib import ExitStack

names = ["sales.csv", "report.txt", "big_regions.csv"]

with ExitStack() as stack:
    files = [stack.enter_context(open(f"context_demo/{n}", encoding="utf-8"))
             for n in names]
    print("opened", len(files), "files at once")
    for name, f in zip(names, files):
        print(f"  {name:18} first line: {f.readline().strip()}")

print("all closed:", all(f.closed for f in files))

opened 3 files at once
  sales.csv          first line: region,units
  report.txt         first line: Regional report
  big_regions.csv    first line: region,units
all closed: True


`stack.enter_context(cm)` enters a manager now and registers its exit for later. When the `with` block ends, everything registered is exited in reverse order.

---
# 13. Common Mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| `__enter__` with no `return self` | `'NoneType' object has no attribute ...` | Return something |
| Cleanup after `yield`, no `try/finally` | Teardown silently skipped when the block raises | Wrap the `yield` |
| `__exit__` returning a truthy non-boolean | Exceptions vanish for no visible reason | Return `True`, `False`, or nothing |
| Reusing a `@contextmanager` object | Cryptic `AttributeError` on the second `with` | Call the function again each time |
| Several statements inside `with suppress(...)` | Later statements silently skipped | One statement per block |
| `with` on a plain value | `object does not support the context manager protocol` | Pass the manager, not the path/number |

One more worth stating plainly: **do not open a file without `with`.** Chapter 10 showed `open()`/`close()` so you would understand what the file object is. From here on there is no reason to write it that way.

---
# 14. Summary: Your Context Manager Cheat Sheet

**What `with` expands to**

```python
manager = EXPR
value = manager.__enter__()       # 'as value'
try:
    body
finally:
    manager.__exit__(exc_type, exc_value, traceback)
```

**The class form**

```python
class Resource:
    def __enter__(self):
        # acquire
        return self               # what 'as' binds
    def __exit__(self, exc_type, exc_value, traceback):
        # release -- always called
        return False              # True would swallow the exception
```

**The generator form**

```python
from contextlib import contextmanager

@contextmanager
def resource():
    # acquire
    try:
        yield value               # what 'as' binds
    finally:
        pass                      # release
```

**`__exit__`'s arguments**

| Block finished | `exc_type` | `exc_value` | `traceback` |
|---|---|---|---|
| Cleanly | `None` | `None` | `None` |
| By raising | the exception class | the exception object | its traceback |

**`contextlib` essentials**

| Tool | Purpose |
|---|---|
| `@contextmanager` | Generator function to context manager |
| `suppress(SomeError)` | Ignore one kind of error (one statement per block) |
| `closing(obj)` | Call `obj.close()` on the way out |
| `redirect_stdout(buf)` | Capture what a block prints |
| `ExitStack()` | A number of managers known only at runtime |

**Several at once**

```python
with open("in.csv") as fin, open("out.csv", "w") as fout:
    ...
```

**When to reach for one:** any time you write "and then remember to put it back" — files, connections, locks, settings, directories, transactions, timers.

---

**Next:** chapter 15 tours the standard library modules you will actually reach for — `collections`, `itertools`, `functools` and `datetime` — including `defaultdict` and `Counter`, which chapter 10 used without ever explaining.